# NYC Taxi Demand — Exploration

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['figure.figsize'] = (12, 4)

In [ ]:
files = sorted(glob.glob('../data/raw/yellow_tripdata_*.csv'))
print(f'Loading {len(files)} files: {[f.split("/")[-1] for f in files]}')

df = pd.concat(
    [pd.read_csv(f, parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime']) for f in files],
    ignore_index=True
)

df['pickup_hour']  = df['tpep_pickup_datetime'].dt.hour
df['pickup_month'] = df['tpep_pickup_datetime'].dt.to_period('M')

print(f'Total rows: {len(df):,}')

## 1. Rows per month

In [ ]:
rows_per_month = df.groupby('pickup_month').size().rename('trip_count')
print(rows_per_month.to_string())

rows_per_month.plot(kind='bar', title='Trips per Month', xlabel='Month', ylabel='Trips')
plt.xticks(rotation=45)
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 2. Columns and dtypes

In [ ]:
dtype_summary = pd.DataFrame({'dtype': df.dtypes, 'non_null': df.notnull().sum(), 'null': df.isnull().sum()})
dtype_summary

## 3. Trip distribution by hour and zone

In [ ]:
# Bin lat/lon into a coarse grid (~1km cells) to approximate zones
# NYC bounding box: lat 40.50–40.92, lon -74.26 to -73.70
lat_bins = pd.interval_range(start=40.50, end=40.92, freq=0.02)
lon_bins = pd.interval_range(start=-74.26, end=-73.70, freq=0.02)

df['lat_zone'] = pd.cut(df['pickup_latitude'],  bins=lat_bins)
df['lon_zone'] = pd.cut(df['pickup_longitude'], bins=lon_bins)
df['zone']     = df['lat_zone'].astype(str) + ' / ' + df['lon_zone'].astype(str)

In [ ]:
# Hourly distribution
hourly = df.groupby('pickup_hour').size()
hourly.plot(kind='bar', title='Trips by Pickup Hour', xlabel='Hour of Day', ylabel='Trips')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 zones by trip count
zone_counts = df.groupby('zone').size().nlargest(20).rename('trip_count')
zone_counts.plot(kind='barh', title='Top 20 Pickup Zones', xlabel='Trips')
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 4. Demand — trips per zone per hour

In [ ]:
demand = (
    df.groupby(['zone', 'pickup_hour'])
    .size()
    .rename('trip_count')
    .reset_index()
)
print(f'Demand table shape: {demand.shape}')
demand.head(10)

In [ ]:
# Hourly demand profile for the top 5 zones
top5_zones = zone_counts.head(5).index.tolist()
top5_demand = demand[demand['zone'].isin(top5_zones)]

fig, ax = plt.subplots()
for zone, grp in top5_demand.groupby('zone'):
    ax.plot(grp['pickup_hour'], grp['trip_count'], marker='o', label=zone[:30])
ax.set_title('Hourly Demand — Top 5 Zones')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Trips')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.show()

## 5. Missing and garbage data

In [ ]:
total = len(df)

checks = {
    'null pickup_datetime':     df['tpep_pickup_datetime'].isnull().sum(),
    'null dropoff_datetime':    df['tpep_dropoff_datetime'].isnull().sum(),
    'null passenger_count':     df['passenger_count'].isnull().sum(),
    'null trip_distance':       df['trip_distance'].isnull().sum(),
    'null pickup_latitude':     df['pickup_latitude'].isnull().sum(),
    'null pickup_longitude':    df['pickup_longitude'].isnull().sum(),
    'zero passenger_count':     (df['passenger_count'] == 0).sum(),
    'negative fare_amount':     (df['fare_amount'] < 0).sum(),
    'zero/negative trip_dist':  (df['trip_distance'] <= 0).sum(),
    'pickup outside NYC bbox':  (
        (df['pickup_latitude']  < 40.50) | (df['pickup_latitude']  > 40.92) |
        (df['pickup_longitude'] < -74.26) | (df['pickup_longitude'] > -73.70)
    ).sum(),
    'dropoff before pickup':    (df['tpep_dropoff_datetime'] <= df['tpep_pickup_datetime']).sum(),
}

quality = pd.DataFrame({
    'issue_count': checks,
    'pct_of_total': {k: round(v / total * 100, 3) for k, v in checks.items()}
})
quality

In [ ]:
quality['pct_of_total'].plot(
    kind='barh',
    title='Data Quality Issues (% of total rows)',
    xlabel='% of rows affected'
)
plt.tight_layout()
plt.show()